## PRT564 Assessment 4 — Classification Models
### Naive Bayes vs SVM vs Random Forest
**Group 15 — Sydney Campus**

In [ ]:
import os

BASE = r"C:\Users\ranas\OneDrive\Desktop\Rana_Research_Workspace\Rana_Research_Workspace"
CLEAN   = os.path.join(BASE, "Clean_Dataset")
CLF_OUT = os.path.join(BASE, "Classification_Outputs")
os.makedirs(CLF_OUT, exist_ok=True)

for p, n in [(CLEAN,"Clean_Dataset"),(CLF_OUT,"Classification_Outputs")]:
    print(n, "found" if os.path.exists(p) else "NOT FOUND")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')
print("libraries loaded")

In [ ]:
df = pd.read_csv(os.path.join(CLEAN, 'classification_dataset.csv'), parse_dates=['quarter'])
print(f"loaded: {df.shape[0]} rows x {df.shape[1]} cols")
print("\nclass balance:")
print(df['cycle'].value_counts())

In [ ]:
FEATURES = ['RPPI_lag1_scaled','RPPI_lag4_scaled','EHPI_YoY_scaled','ADPI_YoY_scaled',
            'MedianHP_YoY_scaled','Transfers_YoY_scaled','ma_4q_scaled','vol_4q_scaled',
            'price_per_dw_scaled','transfer_to_stock_scaled','QoQ_growth_scaled','city_encoded']
TARGET = 'cycle'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"train: {X_train.shape[0]}  |  test: {X_test.shape[0]}")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Model 1 — Naive Bayes (Week 7 — Probabilistic Classifier)

In [ ]:
param_grid_nb = {'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]}

nb_grid = GridSearchCV(GaussianNB(), param_grid_nb, cv=cv, scoring='f1_macro', n_jobs=-1)
nb_grid.fit(X_train, y_train)

print(f"best Naive Bayes params: {nb_grid.best_params_}")
print(f"best CV F1 (macro): {nb_grid.best_score_:.4f}")

nb_model = nb_grid.best_estimator_
y_pred_nb = nb_model.predict(X_test)
y_proba_nb = nb_model.predict_proba(X_test)

print("\n" + "="*60)
print("NAIVE BAYES - TEST RESULTS")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb, average='macro'):.4f}")
print(f"F1 (macro): {f1_score(y_test, y_pred_nb, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_nb, digits=4))

## Model 2 — Support Vector Machine (Week 8 — Linear & Non-linear)

In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1, 1],
    'kernel': ['rbf']
}

svm_grid = GridSearchCV(SVC(probability=True, class_weight='balanced', random_state=42),
                        param_grid_svm, cv=cv, scoring='f1_macro', n_jobs=-1)
svm_grid.fit(X_train, y_train)

print(f"best SVM params: {svm_grid.best_params_}")
print(f"best CV F1 (macro): {svm_grid.best_score_:.4f}")

svm_model = svm_grid.best_estimator_
y_pred_svm = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)

print("\n" + "="*60)
print("SVM (RBF KERNEL) - TEST RESULTS")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_svm, average='macro'):.4f}")
print(f"F1 (macro): {f1_score(y_test, y_pred_svm, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_svm, digits=4))

## Model 3 — Random Forest (Week 10 — Ensemble Learning)

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth':    [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, class_weight='balanced'),
                       param_grid_rf, cv=cv, scoring='f1_macro', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print(f"best Random Forest params: {rf_grid.best_params_}")
print(f"best CV F1 (macro): {rf_grid.best_score_:.4f}")

rf_model = rf_grid.best_estimator_
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

print("\n" + "="*60)
print("RANDOM FOREST - TEST RESULTS")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"F1 (macro): {f1_score(y_test, y_pred_rf, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_rf, digits=4))

## Save Predictions and Outputs

In [ ]:
results = []
for name, m, yp, yprob, cv_score, params in [
    ('NaiveBayes',   nb_model,  y_pred_nb,  y_proba_nb,  nb_grid.best_score_,  nb_grid.best_params_),
    ('SVM',          svm_model, y_pred_svm, y_proba_svm, svm_grid.best_score_, svm_grid.best_params_),
    ('RandomForest', rf_model,  y_pred_rf,  y_proba_rf,  rf_grid.best_score_,  rf_grid.best_params_),
]:
    results.append({
        'Model': name,
        'CV_F1_macro':       cv_score,
        'Test_Accuracy':     accuracy_score(y_test, yp),
        'Test_Precision':    precision_score(y_test, yp, average='macro'),
        'Test_Recall':       recall_score(y_test, yp, average='macro'),
        'Test_F1_macro':     f1_score(y_test, yp, average='macro'),
        'Test_F1_weighted':  f1_score(y_test, yp, average='weighted'),
        'Best_Params':       str(params),
    })

results_df = pd.DataFrame(results)
print(results_df.drop(columns=['Best_Params']).round(4).to_string(index=False))
results_df.to_csv(os.path.join(CLF_OUT, 'classification_results.csv'), index=False)

preds_df = pd.DataFrame({
    'y_true': y_test,
    'pred_NaiveBayes':   y_pred_nb,
    'pred_SVM':          y_pred_svm,
    'pred_RandomForest': y_pred_rf,
})
preds_df.to_csv(os.path.join(CLF_OUT, 'predictions.csv'), index=False)

np.save(os.path.join(CLF_OUT, 'proba_NaiveBayes.npy'),   y_proba_nb)
np.save(os.path.join(CLF_OUT, 'proba_SVM.npy'),          y_proba_svm)
np.save(os.path.join(CLF_OUT, 'proba_RandomForest.npy'), y_proba_rf)
np.save(os.path.join(CLF_OUT, 'classes.npy'),            nb_model.classes_)

fi = pd.DataFrame({'feature': FEATURES, 'importance': rf_model.feature_importances_}).sort_values('importance', ascending=False)
fi.to_csv(os.path.join(CLF_OUT, 'feature_importance.csv'), index=False)

print("\nfeature importance (Random Forest):")
print(fi.head(8).to_string(index=False))
print("\nall classification outputs saved")